# Video-only RVQ-VAE + TF-IDF

Notebook dùng duy nhất đặc trưng ViTPose của bài báo. Chạy `smoke` trước, sau đó đổi thành `full` để đánh giá đủ 9 fold.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/xndung/fidgety/data")
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path("/kaggle/working/Fidgety")
UPLOADED_PROJECT_DIR = None
RUN_MODE = "smoke"
SEED = 42
GPU_WORKERS = 2
CPU_THREADS_PER_WORKER = 2
OUTPUT_DIR = Path(f"/kaggle/working/video_rvq_{RUN_MODE}")

if RUN_MODE == "smoke":
    FOLDS = [1]
    EPOCHS = 5
    PATIENCE = 3
elif RUN_MODE == "full":
    FOLDS = list(range(1, 10))
    EPOCHS = 80
    PATIENCE = 10
else:
    raise ValueError("RUN_MODE must be smoke or full")

## Chuẩn bị source

Nếu chưa đưa thư mục `đồ án` lên repository, hãy upload nó thành một Kaggle Dataset và đặt `UPLOADED_PROJECT_DIR` ở cell trên thành đường dẫn tới thư mục đó.

In [ ]:
import shutil
import subprocess
import sys

if UPLOADED_PROJECT_DIR is not None:
    PROJECT_DIR = Path(UPLOADED_PROJECT_DIR)
else:
    if SOURCE_DIR.exists():
        subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
    PROJECT_DIR = SOURCE_DIR / "đồ án"

if not (PROJECT_DIR / "run.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy project tại {PROJECT_DIR}. Hãy push source lên GitHub hoặc đặt UPLOADED_PROJECT_DIR.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print("Project:", PROJECT_DIR)

In [ ]:
import torch

if not DATA_ROOT.is_dir() and (DATA_ROOT.parent / "data.zip").is_file():
    DATA_ROOT = DATA_ROOT.parent / "data.zip"
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings > Accelerator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_ROOT)
VISIBLE_GPUS = torch.cuda.device_count()
for gpu in range(VISIBLE_GPUS):
    print(f"GPU {gpu}:", torch.cuda.get_device_name(gpu))
print("Parallel GPU workers:", min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS)))
print("Folds:", FOLDS)
print("Epochs:", EPOCHS)

## Huấn luyện và đánh giá

In [ ]:
from collections import deque
import os
import time
import pandas as pd

sys.path.insert(0, str(PROJECT_DIR))
from experiment import summarize

JOBS_DIR = OUTPUT_DIR / "_jobs"
JOBS_DIR.mkdir(parents=True, exist_ok=True)

def fold_complete(fold):
    path = JOBS_DIR / f"fold_{fold}" / "fold_results.csv"
    if not path.is_file():
        return False
    frame = pd.read_csv(path)
    actual = set(zip(frame["fold"], frame["classifier"]))
    return {(fold, "tfidf_logistic"), (fold, "tfidf_mlp")}.issubset(actual)

def fold_command(fold):
    return [
        sys.executable, "-u", str(PROJECT_DIR / "run.py"),
        "--data-path", str(DATA_ROOT),
        "--output-dir", str(JOBS_DIR / f"fold_{fold}"),
        "--folds", str(fold),
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--patience", str(PATIENCE),
        "--batch-size", "64",
        "--patch-size", "10",
        "--hidden-dim", "256",
        "--latent-dim", "64",
        "--codebook-size", "64",
        "--quantizers", "2",
        "--ngram-order", "2",
        "--device", "cuda"
    ]

worker_count = min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS))
pending = deque(fold for fold in FOLDS if not fold_complete(fold))
active = {}
print(f"Starting {worker_count} GPU workers for folds {FOLDS}")
try:
    while pending or active:
        for gpu in [index for index in range(worker_count) if index not in active]:
            if not pending:
                break
            fold = pending.popleft()
            environment = os.environ.copy()
            environment["CUDA_VISIBLE_DEVICES"] = str(gpu)
            environment["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_WORKER)
            environment["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_WORKER)
            process = subprocess.Popen(fold_command(fold), env=environment)
            active[gpu] = (process, fold)
            print(f"GPU {gpu} started fold {fold}, pid={process.pid}", flush=True)
        if active:
            time.sleep(1)
        for gpu, (process, fold) in list(active.items()):
            returncode = process.poll()
            if returncode is None:
                continue
            if returncode:
                raise RuntimeError(f"Fold {fold} failed on GPU {gpu} with code {returncode}")
            print(f"GPU {gpu} completed fold {fold}", flush=True)
            del active[gpu]
except BaseException:
    for process, _ in active.values():
        if process.poll() is None:
            process.terminate()
    for process, _ in active.values():
        process.wait()
    raise

frames = []
for fold in FOLDS:
    job = JOBS_DIR / f"fold_{fold}"
    frame = pd.read_csv(job / "fold_results.csv")
    frames.append(frame[frame["fold"] == fold])
    shutil.copytree(job / f"fold_{fold}", OUTPUT_DIR / f"fold_{fold}", dirs_exist_ok=True)
fold_results = pd.concat(frames, ignore_index=True).sort_values(["classifier", "fold"])
summary = summarize(fold_results)
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(summary.to_string(index=False))

## Kết quả so với baseline video của bài báo

In [ ]:
import pandas as pd
from IPython.display import display

fold_results = pd.read_csv(OUTPUT_DIR / "fold_results.csv")
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
display(fold_results.round(4))
display(summary.round(4))
if RUN_MODE == "smoke":
    print("Smoke mode chỉ kiểm tra pipeline, không dùng để kết luận so sánh với bài báo.")
else:
    print("Published VID balanced accuracy: 90.66%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot = summary.copy()
values = 100 * plot["balanced_accuracy_mean"].to_numpy()
errors = 100 * (plot["balanced_accuracy_ci_high"] - plot["balanced_accuracy_mean"]).to_numpy()
positions = np.arange(len(plot))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions, values, yerr=errors if np.isfinite(errors).all() else None, capsize=5)
ax.axhline(90.6606467, color="black", linestyle="--", label="Published video CNN: 90.66%")
ax.set_xticks(positions, plot["classifier"], rotation=15)
ax.set_ylabel("Balanced accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
plt.show()

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Artifacts:", archive)